In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mdsetup import MDSetup
import pickle

# Change to the correct directory
os.chdir('')

# Initialize MDSetup
lammps_setup = MDSetup(
    system_setup="input_tob_all/setup_mechanical_pcff.yaml",
    simulation_default="input_tob_all/defaults.yaml",
    simulation_ensemble="input_tob_all/ensemble.yaml",
    simulation_sampling="input_tob_all/sampling_mechanical.yaml",
    submission_command="qsub",
)


# Define paths
LJ_SETS = [f"LJ_set_{i+1}" for i in range(50)]
TOB_STRUCTURES = ["Tob9", "Tob11", "Tob11H", "Tob14"]
TOB_STRUCTURES_n = ["Tob11", "Tob11H", "Tob14"]

source_colors = {"Tob9": "blue", "Tob11": "green", "Tob11H": "red", "Tob14": "purple"}

# Ground truth values (None means missing)
ground_truths = {
    "Tob14": {
        "Density": {"EXP": 2.23},
        "SE004": {"Tariq": 635},
        "BM": {"EXP": 47},
    },
    "Tob11": {
        "Density": {"EXP": 2.46},
        "SE004": {"Tariq": 680},
        "BM": {"EXP": 71},
    },
    "Tob11H": {
        "Density": {"EXP": 2.39},
        "SE004": {"Tariq": 325},
        "BM": {"EXP": None},
    },
    "Tob9": {
        "Density": {"EXP": 2.865},
        "SE004": {"Tariq": 805},
        "BM": {"EXP": 115},
    },
}

# Colors for each GT source
source_linestyles = {"DFT": "--", "EXP": "-.", "Tariq": ":"}


In [ ]:
# Define ensemble and analysis parameters
ENSEMBLE = "01_npt"
TIME_FRACTION = 0.2  # Percentage to discard from the beginning of the simulation
PROPERTIES = {
    # "lattice": ["a", "b", "c", "alpha", "beta", "gamma"],
    "density": ["density"],
    # "energy": ["potential energy"],
}

# Storage dictionary for results
den_results = {prop: {tob: [] for tob in TOB_STRUCTURES} for prop in PROPERTIES}

# Loop over LJ parameter sets and Tob structures
for lj_set in LJ_SETS:
    for tob_structure in TOB_STRUCTURES:
        analysis_folder = f"{tob_structure}/{lj_set}/equilibration"
        
        # Store results for the current LJ set
        lj_results = {prop: [] for prop in PROPERTIES}

        for output_suffix, props in PROPERTIES.items():
            extracted_values = lammps_setup.analysis_extract_properties(
                analysis_folder=analysis_folder,
                ensemble=ENSEMBLE,
                extracted_properties=props,
                output_suffix=output_suffix,
                time_fraction=TIME_FRACTION,
            )
            average_values = extracted_values.get(ENSEMBLE, {}).get("data", {}).get("average", {})

            # Extract required values
            for prop in props:
                mean_value = average_values.get(prop, {}).get("mean", None)
                lj_results[output_suffix].append(mean_value)

        # Append results for this LJ set
        for prop in PROPERTIES:
            den_results[prop][tob_structure].append(lj_results[prop])



In [ ]:
# # save den_results in .pkl
# with open("results_den.pkl", "wb") as f:
#     pickle.dump(den_results, f)
    
#load den_results
with open("results_den.pkl", "rb") as f:
    den_results = pickle.load(f)

In [ ]:
plt.figure(figsize=(10, 6))
ax = plt.gca()

for tob_structure in TOB_STRUCTURES:
    data = np.array(den_results['density'][tob_structure])
    if data.size == 0:
        continue
    ax.plot(range(1, len(LJ_SETS) + 1), data[:, 0], label=tob_structure, marker="o", color=source_colors[tob_structure])

    # GT lines for Density
    gt_sources = ground_truths[tob_structure].get("Density", {})
    for source, value in gt_sources.items():
        if value is not None:
            ax.hlines(value, 1, len(LJ_SETS),
                      colors=source_colors[tob_structure],
                      linestyles=source_linestyles[source],
                      label=f"{source} GT ({tob_structure})")

ax.set_xlabel("LJ Set Number")
ax.set_ylabel("Density")
ax.set_title(f"LJ Set vs Density")
ax.set_xticks(range(1, 26))
ax.set_xticklabels([f"S{i+1}" for i in range(25)])
ax.grid()
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()


In [ ]:
# Constants
NA = 6.022e23
CONVERSION = 4184  # kcal/mol to J/mol

# Define structures and LJ param sets


# Box dimensions (structure-dependent)
box_coords = {
    "Tob9":     [-0.378508016, 21.933491984, -0.012474231, 21.896525769],
    "Tob11":    [0.527972175, 23.057572175, -0.431033519, 21.723966481],
    "Tob11H":   [0.065633394, 22.383633394, -0.317852898, 29.242147102],
    "Tob14":    [-0.658063909, 21.871536091, -0.289451837, 21.985548163],
}

# SE results
se_results = {struct: [] for struct in TOB_STRUCTURES}

# Loop over all structures and LJ param sets
for struct in TOB_STRUCTURES:
    for lj in LJ_SETS:
        base = f"{struct}/{lj}/SE"
        
        state = 'bulk'
        folder = f"{base}/{state}"
        extracted_values = lammps_setup.analysis_extract_properties(
            analysis_folder=folder,
            ensemble="00_nvt",
            extracted_properties=["potential energy"],
            output_suffix="energy",
            time_fraction=0.4,
        )
        bulk_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})

        state = 'vacuum'
        folder = f"{base}/{state}"
        extracted_values = lammps_setup.analysis_extract_properties(
            analysis_folder=folder,
            ensemble="00_nvt",
            extracted_properties=["potential energy"],
            output_suffix="energy",
            time_fraction=0.4,
        )
        vac_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})
        
        E_bulk, SD_bulk = bulk_energy["mean"], bulk_energy["std"]
        E_vac, SD_vac = vac_energy["mean"], vac_energy["std"]

        # Compute surface area
        xlo, xhi, ylo, yhi = box_coords[struct]
        A = abs(xhi - xlo) * abs(yhi - ylo) * 1e-20  # m²

        # Compute SE and std dev
        deltaE = 1000 * (E_vac - E_bulk) * CONVERSION / (2 * A * NA)  # mJ/m²
        s_dev = 1000 * (SD_bulk + SD_vac) * CONVERSION / (2 * A * NA)

        se_results[struct].append((deltaE, s_dev))


In [ ]:
# # save se_results in .pkl
# with open("results_se.pkl", "wb") as f:
#     pickle.dump(se_results, f)
    
#load se_results
with open("results_se.pkl", "rb") as f:
    se_results = pickle.load(f)

In [ ]:
plt.figure(figsize=(10, 6))
ax = plt.gca()

for tob_structure in TOB_STRUCTURES:
    SE_vals = [v[0] for v in se_results[tob_structure]]
    errors = [v[1] for v in se_results[tob_structure]]
    # ax.errorbar(range(1, 6), SE_vals, yerr=errors, label=tob_structure, marker='o', capsize=5)
    ax.plot(range(1, len(LJ_SETS)+1), SE_vals, marker='o', label=tob_structure, color=source_colors[tob_structure])

    # GT lines for SE004
    gt_sources = ground_truths[tob_structure].get("SE004", {})
    for source, value in gt_sources.items():
        if value is not None:
            ax.hlines(value, 1, len(LJ_SETS),
                      colors=source_colors[tob_structure],
                      linestyles=source_linestyles[source],
                      label=f"{source} GT ({tob_structure})")

ax.set_title("LJ Parameter Set vs Surface Energy")
ax.set_xlabel("LJ Set Number")
ax.set_ylabel("Surface Energy (mJ/m²)")
ax.set_xticks(range(1, 26))
ax.set_xticklabels([f"S{i+1}" for i in range(25)])
ax.grid(True)
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
ax = plt.gca()

for tob_structure in TOB_STRUCTURES_n:
    SE_vals = [v[0] for v in se_results[tob_structure]]
    errors = [v[1] for v in se_results[tob_structure]]
    # ax.errorbar(range(1, 6), SE_vals, yerr=errors, label=tob_structure, marker='o', capsize=5)
    ax.plot(range(1, len(LJ_SETS)+1), SE_vals, marker='o', label=tob_structure, color=source_colors[tob_structure])

    # GT lines for SE004
    gt_sources = ground_truths[tob_structure].get("SE004", {})
    for source, value in gt_sources.items():
        if value is not None:
            ax.hlines(value, 1, len(LJ_SETS),
                      colors=source_colors[tob_structure],
                      linestyles=source_linestyles[source],
                      label=f"{source} GT ({tob_structure})")

ax.set_title("LJ Parameter Set vs Surface Energy")
ax.set_xlabel("LJ Set Number")
ax.set_ylabel("Surface Energy (mJ/m²)")
ax.set_xticks(range(1, 26))
ax.set_xticklabels([f"S{i+1}" for i in range(25)])
ax.grid(True)
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()


In [ ]:
# Define parameters for deformation analysis
ENSEMBLE = "00_nvt"
DEFORMATION_RATES = [-0.02, -0.01, 0.00, 0.01, 0.02]
TIME_FRACTION = 0.4
METHOD = "VRH"
VISUALIZE_STRESS_STRAIN = False  # Change to True if you want plots per simulation

# Dictionary to hold Bulk Modulus values
bm_results = {tob: [] for tob in TOB_STRUCTURES}
cij_results = {tob: [] for tob in TOB_STRUCTURES}
# Loop through each LJ set and Tob structure
for lj_set in LJ_SETS:
    for tob_structure in TOB_STRUCTURES:
        # Define the deformation analysis folder
        analysis_folder = f"{tob_structure}/{lj_set}/deformation"

        # Run analysis
        BM, Cij = lammps_setup.analysis_mechanical_proerties(
            analysis_folder=analysis_folder,
            ensemble=ENSEMBLE,
            deformation_rates=DEFORMATION_RATES,
            method=METHOD,
            time_fraction=TIME_FRACTION,
            visualize_stress_strain=VISUALIZE_STRESS_STRAIN,
        )

        print(f"✅ Analyzed {tob_structure} with {lj_set}: BM = {BM:.2f} GPa")
        bm_results[tob_structure].append(BM)
        cij_results[tob_structure].append(Cij)



In [ ]:
# order = [0, 5, 6, 7, 8, 9, 1, 10, 11, 12, 13, 14, 2, 15, 16, 17, 18, 19, 3, 20, 21, 22, 23, 24, 4]

# r_tob9 = bm_results['Tob9']
# r_tob9_n = [r_tob9[i] for i in order]
# r_tob11 = bm_results['Tob11']
# r_tob11_n = [r_tob11[i] for i in order]
# r_tob11H = bm_results['Tob11H']
# r_tob11H_n = [r_tob11H[i] for i in order]
# r_tob14 = bm_results['Tob14']
# r_tob14_n = [r_tob14[i] for i in order]

# bm_results = {'Tob9': r_tob9_n, 'Tob11': r_tob11_n, 'Tob11H': r_tob11H_n, 'Tob14': r_tob14_n}

In [ ]:
# # save bm_results and cij_results in .pkl
# with open('results_bm.pkl', 'wb') as f:
#     pickle.dump(bm_results, f)

# with open('results_cij.pkl', 'wb') as f:
#     pickle.dump(cij_results, f)

# Load bm_results and cij_results from .pkl
with open('results_bm.pkl', 'rb') as f:
    bm_results = pickle.load(f)

with open('results_cij.pkl', 'rb') as f:
    cij_results = pickle.load(f)
    

In [ ]:
plt.figure(figsize=(10, 6))
ax = plt.gca()

for tob_structure in TOB_STRUCTURES_n:
    ax.plot(range(1, len(LJ_SETS)+1), bm_results[tob_structure], marker='o', label=tob_structure, color=source_colors[tob_structure])

    # GT lines for BM
    gt_sources = ground_truths[tob_structure].get("BM", {})
    for source, value in gt_sources.items():
        if value is not None:
            ax.hlines(value, 1, len(LJ_SETS),
                      colors=source_colors[tob_structure],
                      linestyles=source_linestyles[source],
                      label=f"{source} GT ({tob_structure})")

ax.set_xlabel("LJ Set Number")
ax.set_ylabel("Bulk Modulus (GPa)")
ax.set_title("LJ Set vs Bulk Modulus")
ax.set_xticks(range(1, 26))
ax.set_xticklabels([f"S{i+1}" for i in range(25)])
ax.grid(True)
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()



In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 18))
axs[0].set_title("C11")
axs[1].set_title("C22")
axs[2].set_title("C33")

for tob_structure in TOB_STRUCTURES_n:
    C11 = [i[0][0] for i in cij_results[tob_structure]]
    C22 = [i[1][1] for i in cij_results[tob_structure]]
    C33 = [i[2][2] for i in cij_results[tob_structure]]
    C = [C11, C22, C33]
    for i in range(3):
    
        axs[i].plot(range(1, len(LJ_SETS)+1), C[i], marker='o', label=tob_structure, color=source_colors[tob_structure])

        axs[i].set_xlabel("LJ Set Number")
        axs[i].set_ylabel("Bulk Modulus (GPa)")
        axs[i].set_xticks(range(1, 26))
        axs[i].set_xticklabels([f"S{i+1}" for i in range(25)])
        axs[i].grid(True)
        handles, labels = axs[i].get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        axs[i].legend(unique.values(), unique.keys(), loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()
